# Sceptical base-rate benchmark

Runs all **54** conditions from `data/base_rate/benchmark.csv`: 9 vignettes × 6 variants (open / MC numeric / MC full × with / without stated probabilities).

**Kaggle setup:** Add-ons → Secrets → `GITHUB_TOKEN` (GitHub PAT, toggled ON). Settings → Internet ON. **Run the setup cell below first.**

**Publishing:** Run all cells through `%choose`.

In [ ]:
import csv
import io
import shutil
import sys
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

GITHUB_OWNER = "FrancisGanong-N"
GITHUB_REPO = "sceptical_llms"
KAGGLE_REPO_DIR = Path("/kaggle/working") / GITHUB_REPO
EXPECTED_BENCHMARK_PROMPTS = 54
FORCE_REPO_REFRESH = False


def benchmark_prompt_count(root: Path) -> int:
    benchmark_csv = root / "data" / "base_rate" / "benchmark.csv"
    if not benchmark_csv.is_file():
        return 0
    with benchmark_csv.open(newline="", encoding="utf-8") as handle:
        return sum(1 for _ in csv.DictReader(handle))


def has_fresh_repo(root: Path) -> bool:
    return (
        (root / "benchmarks" / "base_rate_tasks.py").is_file()
        and benchmark_prompt_count(root) >= EXPECTED_BENCHMARK_PROMPTS
    )


def download_repo_from_github() -> Path:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN").strip()
    url = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/zipball/main"
    request = urllib.request.Request(
        url,
        headers={
            "Authorization": f"Bearer {token}",
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28",
            "User-Agent": "kaggle-sceptical-llms-benchmark",
        },
    )

    staging = Path("/kaggle/working") / "_repo_staging"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir()

    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            with zipfile.ZipFile(io.BytesIO(response.read())) as archive:
                archive.extractall(staging)
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"GitHub download failed ({exc.code}): {body[:300]}") from exc

    extracted = next(p for p in staging.iterdir() if p.is_dir())
    if KAGGLE_REPO_DIR.exists():
        shutil.rmtree(KAGGLE_REPO_DIR)
    shutil.copytree(extracted, KAGGLE_REPO_DIR)
    shutil.rmtree(staging)
    return KAGGLE_REPO_DIR


def bootstrap_repo() -> Path:
    if not FORCE_REPO_REFRESH and has_fresh_repo(KAGGLE_REPO_DIR):
        return KAGGLE_REPO_DIR

    for candidate in (Path.cwd(), Path.cwd().parent):
        if has_fresh_repo(candidate):
            return candidate

    if not Path("/kaggle/working").is_dir():
        raise RuntimeError(
            "Could not find a fresh sceptical-llms repo (need "
            f"{EXPECTED_BENCHMARK_PROMPTS} rows in benchmark.csv). Run from the repo, "
            "or on Kaggle set GITHUB_TOKEN and enable Internet, then re-run this cell."
        )

    return download_repo_from_github()


ROOT = bootstrap_repo()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repo root:", ROOT)
print("Benchmark rows:", benchmark_prompt_count(ROOT))

In [ ]:
import kaggle_benchmarks as kbench
from benchmarks.base_rate_tasks import base_rate_normative_accuracy

run = base_rate_normative_accuracy.run(llm=kbench.llm)
print("Normative accuracy:", run.result)

In [ ]:
%choose base_rate_normative_accuracy